# 01 — Dataset & Exploration des données (EDA)Ce notebook couvre la partie **Personne 1** :- vérification de la structure du dataset (train/val/test + NORMAL/PNEUMONIA)- comptage des images par classe et par split- analyse des dimensions d'images- distribution des formats (jpg/png/...)- détection d'images corrompues- visualisation d'exemples d'imagesLes figures sont enregistrées dans `../outputs/figures/`.

In [ ]:
from pathlib import Pathfrom collections import Counterimport randomimport numpy as npimport pandas as pdfrom PIL import Image, UnidentifiedImageErrorimport matplotlib.pyplot as pltimport seaborn as snssns.set_theme(style="whitegrid")random.seed(42)np.random.seed(42)SPLITS = ["train", "val", "test"]CLASSES = ["NORMAL", "PNEUMONIA"]VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}FIGURES_DIR = (Path.cwd() / ".." / "outputs" / "figures").resolve()FIGURES_DIR.mkdir(parents=True, exist_ok=True)print(f"Figures sauvegardées dans: {FIGURES_DIR}")

In [ ]:
def is_valid_dataset_root(path: Path) -> bool:    return all((path / split / cls).exists() for split in SPLITS for cls in CLASSES)def find_dataset_root() -> Path:    cwd = Path.cwd().resolve()    bases = [cwd, cwd.parent, cwd.parent.parent]    candidates = []    for base in bases:        candidates.extend([            base,            base / "chest_xray",            base / "data" / "chest_xray",            base / "dataset" / "chest_xray",            base / "__MACOSX" / "chest_xray",        ])    seen = set()    unique_candidates = []    for cand in candidates:        cand = cand.resolve()        if cand not in seen:            unique_candidates.append(cand)            seen.add(cand)    valid_roots = [c for c in unique_candidates if c.exists() and is_valid_dataset_root(c)]    if not valid_roots:        raise FileNotFoundError(            "Aucune racine dataset valide trouvée (attendu: split/class). "            "Vérifie l'emplacement du dossier chest_xray."        )    def image_count(root: Path) -> int:        total = 0        for split in SPLITS:            for cls in CLASSES:                total += sum(1 for p in (root / split / cls).iterdir() if p.is_file())        return total    best_root = max(valid_roots, key=image_count)    return best_rootDATASET_ROOT = find_dataset_root()print(f"Dataset root détecté: {DATASET_ROOT}")for split in SPLITS:    for cls in CLASSES:        path = DATASET_ROOT / split / cls        print(f"- {split}/{cls}: {'OK' if path.exists() else 'MISSING'}")

In [ ]:
rows = []all_image_paths = []format_counter = Counter()for split in SPLITS:    for cls in CLASSES:        class_dir = DATASET_ROOT / split / cls        files = [p for p in class_dir.iterdir() if p.is_file()]        for p in files:            ext = p.suffix.lower()            format_counter[ext if ext else "<no_ext>"] += 1            all_image_paths.append((p, split, cls))        rows.append({            "split": split,            "class": cls,            "count": len(files),        })counts_df = pd.DataFrame(rows)counts_pivot = counts_df.pivot(index="split", columns="class", values="count").fillna(0).astype(int)display(counts_df.sort_values(["split", "class"]).reset_index(drop=True))print("\nTableau pivot (split x class):")display(counts_pivot)class_totals = counts_df.groupby("class")["count"].sum().reindex(CLASSES)print("Totaux par classe:")display(class_totals.to_frame("total_images"))if class_totals.min() > 0:    imbalance_ratio = class_totals.max() / class_totals.min()    print(f"Ratio de déséquilibre (max/min): {imbalance_ratio:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))plot_df = counts_df.groupby("class", as_index=False)["count"].sum()sns.barplot(data=plot_df, x="class", y="count", hue="class", legend=False, ax=ax)ax.set_title("Nombre total d'images par classe")ax.set_xlabel("Classe")ax.set_ylabel("Nombre d'images")for p in ax.patches:    ax.annotate(        f"{int(p.get_height())}",        (p.get_x() + p.get_width() / 2, p.get_height()),        ha="center",        va="bottom",        fontsize=10    )fig.tight_layout()counts_fig_path = FIGURES_DIR / "eda_class_counts.png"fig.savefig(counts_fig_path, dpi=200, bbox_inches="tight")plt.show()print(f"Figure enregistrée: {counts_fig_path}")

In [ ]:
format_df = pd.DataFrame(sorted(format_counter.items()), columns=["extension", "count"])format_df = format_df.sort_values("count", ascending=False).reset_index(drop=True)display(format_df)valid_format_df = format_df[format_df["extension"].isin(VALID_EXTS)]other_format_df = format_df[~format_df["extension"].isin(VALID_EXTS)]if len(other_format_df) > 0:    print("Formats inattendus détectés:")    display(other_format_df)else:    print("Aucun format inattendu détecté.")

In [ ]:
corrupted = []for path, split, cls in all_image_paths:    try:        with Image.open(path) as img:            img.verify()    except (UnidentifiedImageError, OSError, ValueError) as exc:        corrupted.append({            "path": str(path),            "split": split,            "class": cls,            "error": str(exc),        })corrupted_df = pd.DataFrame(corrupted)print(f"Nombre d'images corrompues: {len(corrupted_df)}")if len(corrupted_df) > 0:    display(corrupted_df.head(20))

In [ ]:
dimension_rows = []for path, split, cls in all_image_paths:    try:        with Image.open(path) as img:            width, height = img.size        dimension_rows.append({            "path": str(path),            "split": split,            "class": cls,            "width": width,            "height": height,            "area": width * height,        })    except Exception:        continuedim_df = pd.DataFrame(dimension_rows)print(f"Images analysées pour dimensions: {len(dim_df)}")summary_dims = dim_df[["width", "height"]].describe().Tdisplay(summary_dims)fig, axes = plt.subplots(1, 2, figsize=(12, 4))sns.histplot(dim_df["width"], bins=30, ax=axes[0], color="steelblue")axes[0].set_title("Distribution des largeurs")axes[0].set_xlabel("Width")sns.histplot(dim_df["height"], bins=30, ax=axes[1], color="darkorange")axes[1].set_title("Distribution des hauteurs")axes[1].set_xlabel("Height")fig.tight_layout()dims_fig_path = FIGURES_DIR / "eda_dimensions_hist.png"fig.savefig(dims_fig_path, dpi=200, bbox_inches="tight")plt.show()print(f"Figure enregistrée: {dims_fig_path}")

In [ ]:
def sample_images_by_class(target_class: str, n: int = 5):    preferred_order = ["train", "val", "test"]    collected = []    for split in preferred_order:        folder = DATASET_ROOT / split / target_class        imgs = [p for p in folder.iterdir() if p.is_file()]        random.shuffle(imgs)        for p in imgs:            collected.append((p, split))            if len(collected) >= n:                return collected    return collectedn_per_class = 5normal_samples = sample_images_by_class("NORMAL", n=n_per_class)pneumonia_samples = sample_images_by_class("PNEUMONIA", n=n_per_class)n_cols = max(len(normal_samples), len(pneumonia_samples), 1)fig, axes = plt.subplots(2, n_cols, figsize=(3 * n_cols, 6))if n_cols == 1:    axes = np.array(axes).reshape(2, 1)for i in range(n_cols):    axes[0, i].axis("off")    axes[1, i].axis("off")for i, (img_path, split) in enumerate(normal_samples):    with Image.open(img_path) as img:        axes[0, i].imshow(img.convert("L"), cmap="gray")    axes[0, i].set_title(f"NORMAL ({split})")    axes[0, i].axis("off")for i, (img_path, split) in enumerate(pneumonia_samples):    with Image.open(img_path) as img:        axes[1, i].imshow(img.convert("L"), cmap="gray")    axes[1, i].set_title(f"PNEUMONIA ({split})")    axes[1, i].axis("off")fig.suptitle("Exemples d'images: NORMAL vs PNEUMONIA", fontsize=14)fig.tight_layout()samples_fig_path = FIGURES_DIR / "eda_samples_normal_vs_pneumonia.png"fig.savefig(samples_fig_path, dpi=200, bbox_inches="tight")plt.show()print(f"Figure enregistrée: {samples_fig_path}")

## Points à reporter dans le rapportAprès exécution, utilise les sorties de ce notebook pour remplir les sections:1. **Dataset** : source Kaggle, structure `train/val/test` et classes.2. **Exploration des données** : histogramme des classes, dimensions, formats.3. **Qualité et limites** : déséquilibre, images corrompues, variabilité des résolutions.Figures principales générées:- `eda_class_counts.png`- `eda_samples_normal_vs_pneumonia.png`- `eda_dimensions_hist.png`